<a href="https://colab.research.google.com/github/JCF-the-creator/Multi_Motorsport_Analytics_Desempenho_e_Estrategia/blob/main/Codigo_PY_MotoGP/MotoGp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 72.2 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [6]:
import os
import requests
from pathlib import Path
from genericpath import exists
import pdfplumber
import pandas as pd
import zipfile
import json
import re

HEADERS = {"User-Agent": "Mozilla/5.0"}

#anos
YEARS1 = [2002, 2003, 2004, 2005, 2006, 2007, 2008]

#MUDA A POSIÇÃO DO LINK
YEARS2 = [2009, 2010,
        2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019,
        2020, 2021, 2022, 2023, 2024, 2025, 2026]

# Lista de circuitos (slugs oficiais usados nos PDFs)
CIRCUITS = ["VAL","CAT","BRNO","MIS","SEP","QAT","AUS","ARG","JPN","ITA","GER","NED","GBR","USA","POR","ARA","SMR","MAL","FRA"]

#etapas
ETAPAS = ["RAC","QP","Q1","Q2","QP1","QP2"]

#muda com alguns anos
DOC = ["classification","Classification"]

for y in YEARS1:
  for c in CIRCUITS:
    for e in ETAPAS:
      for d in DOC:

          url = f"https://resources.motogp.com/files/results/{y}/MotoGP/{c}/{e}/{d}.pdf"

          pasta_destino = Path(f"motogp/{str(y)}/{c}")
          pasta_destino.mkdir(parents=True, exist_ok=True)
          nome_arquivo = f"{str(y)} {c} {e} {d}.pdf"

          # Cria o caminho completo
          caminho_completo = pasta_destino / nome_arquivo  # O "/" junta os caminhos


          r = requests.get(url, headers=HEADERS, timeout=10)
          if r.status_code == 200:
              with open(caminho_completo, "wb") as f:
                  f.write(r.content)
              print(f"[SAVED] {caminho_completo}")
          else:
              print(f"[NO PDF] {url}")

for y2 in YEARS2:
  for c2 in CIRCUITS:
    for e2 in ETAPAS:
      for d2 in DOC:

          url2 = f"https://resources.motogp.com/files/results/{y2}/{c2}/MotoGP/{e2}/{d2}.pdf"

          pasta_destino = Path(f"motogp/{str(y2)}/{c2}")
          pasta_destino.mkdir(parents=True, exist_ok=True)
          nome_arquivo = f"{str(y2)} {c2} {e2} {d2}.pdf"
          caminho_completo = pasta_destino / nome_arquivo

          r2 = requests.get(url2, headers=HEADERS, timeout=10)
          if r2.status_code == 200:
              with open(caminho_completo, "wb") as f:
                  f.write(r2.content)
              print(f"[SAVED] {caminho_completo}")
          else:
              print(f"[NO PDF] {url}")

# Cria um objeto Path para uma pasta
pastaPrincipal = Path("/content/motogp")

anos = [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
        2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019,
        2020, 2021, 2022, 2023, 2024, 2025, 2026]


for corrida in CIRCUITS:
  for s in anos:
    pastaAVerificar = pastaPrincipal/str(s)/str(corrida)

    try:
      os.rmdir(pastaAVerificar)
    except:
      continue


ROOT = Path("/content/motogp")

def extrair_dados(pdf_path):
    classification = []
    extras_not_classified = []
    extras_not_finished = []
    stats = {}
    ano = pdf_path.parts[-3]
    circuito = pdf_path.parts[-2]

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            texto = page.extract_text()
            if texto:
                for linha in texto.split("\n"):
                    # Classificação principal (Pos, Points, Number)
                    padrao = re.match(
                        r"^(\d+)\s+(\d*)\s+(\d+)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([A-Z]{3})\s+(.+?)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([\d'’:.]+)\s+([\d.]+|\d+\s+laps|Not finished.*)\s*(.*)$",
                        linha
                    )
                    if padrao:
                        pos, points, number, rider, nation, team, moto, total_time, kmh, gap = padrao.groups()
                        if points == "":
                            points = ""  # valor vazio para quem não tem pontuação
                        classification.append({
                            "Year": ano,
                            "Circuit": circuito,
                            "Pos": pos.strip(),
                            "Points": points.strip(),
                            "Number": number.strip(),
                            "Rider": rider.strip(),
                            "Nation": nation.strip(),
                            "Team": team.strip(),
                            "Motorcycle": moto.strip(),
                            "Total Time": total_time.strip(),
                            "Km/h": kmh.strip(),
                            "Gap": gap.strip()
                        })
                        continue

                    # Extras (Not Classified)
                    padrao_nc = re.match(
                        r"^(\d+)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([A-Z]{3})\s+(.+?)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([\d'’:.]+)\s+([\d.]+)\s+(.*)$",
                        linha
                    )
                    if padrao_nc:
                        number, rider, nation, team, moto, total_time, kmh, gap = padrao_nc.groups()
                        extras_not_classified.append({
                            "Number": number.strip(),
                            "Rider": rider.strip(),
                            "Nation": nation.strip(),
                            "Team": team.strip(),
                            "Motorcycle": moto.strip(),
                            "Total Time": total_time.strip(),
                            "Km/h": kmh.strip(),
                            "Gap": gap.strip()
                        })
                        continue

                    # Extras (Not finished first lap)
                    padrao_nf = re.match(
                        r"^(\d+)\s+([A-Za-zÀ-ÿ'\- ]+)\s+([A-Z]{3})\s+(.+?)\s+([A-Za-zÀ-ÿ'\- ]+)$",
                        linha
                    )
                    if padrao_nf:
                        number, rider, nation, team, moto = padrao_nf.groups()
                        extras_not_finished.append({
                            "Number": number.strip(),
                            "Rider": rider.strip(),
                            "Nation": nation.strip(),
                            "Team": team.strip(),
                            "Motorcycle": moto.strip()
                        })
                        continue

                    # Estatísticas extras
                    if "Pole Position" in linha:
                        stats["Pole Position"] = linha
                    elif "Fastest Lap" in linha:
                        stats["Fastest Lap"] = linha
                    elif "Circuit Record Lap" in linha:
                        stats["Circuit Record Lap"] = linha
                    elif "Circuit Best Lap" in linha:
                        stats["Circuit Best Lap"] = linha

    # Estrutura final
    corrida = {
        "Year": ano,
        "Circuit": circuito,
        "Classification": classification,
        "Extras": {
            "Not Classified": extras_not_classified,
            "Not Finished First Lap": extras_not_finished
        },
        "Stats": stats
    }

    # salvar JSON único
    json_path = pdf_path.with_suffix(".full.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(corrida, f, indent=4, ensure_ascii=False)
    print(f"[FULL JSON SAVED] {json_path}")

    # salvar CSV da classificação
    if classification:
        df = pd.DataFrame(classification)
        csv_path = pdf_path.with_suffix(".full.csv")
        df.to_csv(csv_path, index=False)
        print(f"[FULL CSV SAVED] {csv_path}")

    return classification

# Consolidado
all_data = []
for pdf_file in ROOT.rglob("*.pdf"):
    classificacao = extrair_dados(pdf_file)
    if classificacao:
        all_data.extend(classificacao)

# salvar mestre CSV/JSON
if all_data:
    master_df = pd.DataFrame(all_data)
    master_df.to_csv("motogp_master_full.csv", index=False)
    with open("motogp_master_full.json", "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=4, ensure_ascii=False)
    print("[MASTER CREATED] motogp_master_full.csv, motogp_master_full.json")

# Compactar tudo
final_zip = Path("motogp_total.zip")
with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in ROOT.rglob("*"):
        if file_path.is_file():
            arcname = file_path.relative_to(ROOT.parent)
            zipf.write(file_path, arcname)
    zipf.write("motogp_master_full.csv")
    zipf.write("motogp_master_full.json")

print(f"[FINAL ZIP CREATED] {final_zip}")

[SAVED] motogp/2002/VAL/2002 VAL RAC classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/RAC/Classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/QP/classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/QP/Classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/Q1/classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/Q1/Classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/Q2/classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/Q2/Classification.pdf
[SAVED] motogp/2002/VAL/2002 VAL QP1 classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/QP1/Classification.pdf
[SAVED] motogp/2002/VAL/2002 VAL QP2 classification.pdf
[NO PDF] https://resources.motogp.com/files/results/2002/MotoGP/VAL/QP2/Classification.pdf
[SAVED] motogp/2002

In [24]:
import json
import pandas as pd
import numpy as np
from google.colab import files # Biblioteca nativa do Colab para downloads

nome_do_arquivo = 'motogp_master_full.json'

def gerar_e_baixar_csvs(caminho_arquivo):
    print("Lendo o arquivo JSON...")
    try:
        with open(caminho_arquivo, 'r', encoding='utf-8') as f:
            dados = json.load(f)
    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado. Faça o upload novamente na aba da esquerda.")
        return

    lista_classification = []

    print("Processando os dados...")
    for corrida in dados:
        year = corrida.get('Year', '')
        circuit = corrida.get('Circuit', '')


        # 2. Classification
        for piloto in corrida.get('Classification', []):
            linha_piloto = piloto.copy()
            linha_piloto['Year'] = year
            linha_piloto['Circuit'] = circuit
            lista_classification.append(linha_piloto)

    print("Gerando os arquivos CSV...")
    pd.DataFrame(lista_classification).to_csv('Classification.csv', index=False)


    print("\nIniciando o download automático para o seu computador...")
    arquivos_gerados = ['Classification.csv']

    # Esta parte força o navegador a baixar os arquivos
    for arq in arquivos_gerados:
        try:
            files.download(arq)
            print(f"✔️ Baixando: {arq}")
        except Exception as e:
            print(f"❌ Erro ao tentar baixar o arquivo {arq}: {e}")

# Executa o script
gerar_e_baixar_csvs(nome_do_arquivo)

df = pd.read_csv('/content/motogp_master_full.csv')

#todos os valores nulos são 0
df = df.fillna(0)
df = df.replace(['nan', 'NaN', 'None', ''], 0)

#corrige o tempo
df['Total Time'] = df['Total Time'].astype(str).str.replace(r"'", ':', regex=True).str.strip()
df['Total Time'] = df['Total Time'].astype(str).str.replace(r"\.", ':', regex=True).str.strip()

'''
#remove as strings e converte o tempo
df['Gap'] = df['Gap'].astype(str).str.replace(r"'", ':', regex=True).str.strip()
df['Gap'] = df['Gap'].astype(str).str.replace(r"\.", ':', regex=True).str.strip()

df['Gap'] = df['Gap'].astype(str).str.replace(r"\ lap", '', regex=True).str.strip()
'''

#removi a coluna Gap dp CSV por conta do 1 lap que buga o código
#para adicionar a uma nova coluna GAP é necessário converter a ["Total"] de String para Time
#usando pd.to_timedelta
#depois verificando os valores de POS if ["Year"] and ["Circuit"] da linha anterior for igual a essa linha
#ele faz o calculo do total - total anterior
#se falso, ele retorna 00:00:00 por que ae trocou a corrida então ele é o primeiro dela

#a logica acima funciona se as posições estiverem organizada de 1->15 se o csv misturar as posições quebra o codigo

#a coluna GAP pode ser adicionada no POWERBI futuramente

df = df.drop(columns=['Gap'])

df.to_csv('csv_MotoGP_Corrigido.csv', index=False)
files.download('csv_MotoGP_Corrigido.csv')

Lendo o arquivo JSON...
Processando os dados...
Gerando os arquivos CSV...

Iniciando o download automático para o seu computador...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔️ Baixando: Classification.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>